In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from cryptosense.data import load_processed
from cryptosense.features import add_features

DATA_DIR = Path.cwd().parent / "data"
df = add_features(load_processed("BTC/USDT", DATA_DIR))

In [2]:
df["target_rv_5"]=np.sqrt((df["log_return"] ** 2).rolling(5).mean()).shift(-5)

In [3]:
df[["log_return", "target_rv_5"]].tail(7)

,log_return,target_rv_5
date,,
2026-10-10 00:00:00+00:00,0.004260,0.015851
2026-10-11 00:00:00+00:00,0.009412,0.015665
2026-10-12 00:00:00+00:00,-0.023224,NaN
2026-10-13 00:00:00+00:00,0.015986,NaN
2026-10-14 00:00:00+00:00,-0.018041,NaN
2026-10-15 00:00:00+00:00,0.006880,NaN
2026-10-16 00:00:00+00:00,-0.007693,NaN


In [4]:
df[["vol_20", "target_rv_5"]].describe()


,vol_20,target_rv_5
count,3328.000000,3343.000000
mean,0.031460,0.029251
std,0.015893,0.019747
min,0.005908,0.002167
25%,0.020780,0.015821
50%,0.028445,0.024694
75%,0.037938,0.037413
max,0.133058,0.239300


In [5]:
FEATURES = ["log_return", "mom_5", "mom_20", "dist_sma_20", "dist_sma_100",
            "vol_20", "vol_60", "rel_volume_20", "er_20"]
HORIZON = 5

df["base_rv_5"] = np.sqrt((df["log_return"] ** 2).rolling(HORIZON).mean())

data = df.dropna(subset=FEATURES + ["target_rv_5", "base_rv_5"])

train = data.loc[:"2021-12-31"].iloc[:-HORIZON]
val = data.loc["2022-01-01":"2023-12-31"].iloc[:-HORIZON]
test = data.loc["2024-01-01":]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5} {part.index[0].date()} -> {part.index[-1].date()} | {len(part)} days")

day = pd.Timedelta(days=1)
assert train.index[-1] + HORIZON * day <= pd.Timestamp("2021-12-31", tz="UTC")
assert val.index[-1] + HORIZON * day <= pd.Timestamp("2023-12-31", tz="UTC")
print("purge check passed")

train 2017-11-24 -> 2021-12-26 | 1494 days
val   2022-01-01 -> 2023-12-26 | 725 days
test  2024-01-01 -> 2026-10-11 | 1015 days
purge check passed


In [6]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

y_val = val["target_rv_5"]

baselines = {
    "constant": np.full(len(val), train["target_rv_5"].mean()),
    "vol_20": val["vol_20"],
    "base_rv_5": val["base_rv_5"],
}

rows = []
for name, forecast in baselines.items():
    rows.append({
        "baseline": name,
        "MAE": mean_absolute_error(y_val, forecast),
        "RMSE": root_mean_squared_error(y_val, forecast),
    })

baseline_results = pd.DataFrame(rows).set_index("baseline")
baseline_results.round(5)

,MAE,RMSE
baseline,,
constant,0.01673,0.01925
vol_20,0.01314,0.01723
base_rv_5,0.01429,0.01922


In [7]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=20,
    random_state=0,
    n_jobs=-1,
)
rf.fit(train[FEATURES], train["target_rv_5"])
rf_pred = rf.predict(val[FEATURES])

baseline_results.loc["random_forest"] = [
    mean_absolute_error(y_val, rf_pred),
    root_mean_squared_error(y_val, rf_pred),
]
baseline_results.round(5)

,MAE,RMSE
baseline,,
constant,0.01673,0.01925
vol_20,0.01314,0.01723
base_rv_5,0.01429,0.01922
random_forest,0.01325,0.01697


In [8]:
pd.Series({
    "actual (val)": y_val.mean(),
    "vol_20 forecast": val["vol_20"].mean(),
    "random_forest forecast": rf_pred.mean(),
    "train target mean": train["target_rv_5"].mean(),
}).round(5)

actual (val)              0.02440
vol_20 forecast           0.02690
random_forest forecast    0.02945
train target mean         0.03579
dtype: float64

In [9]:
y_train_rel = np.log(train["target_rv_5"] / train["vol_20"])

scale = np.exp(y_train_rel.mean())
vol_20_scaled = val["vol_20"] * scale

rf_rel = RandomForestRegressor(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)
rf_rel.fit(train[FEATURES], y_train_rel)
rf_rel_pred = val["vol_20"] * np.exp(rf_rel.predict(val[FEATURES]))

for name, pred in [("vol_20_scaled", vol_20_scaled), ("rf_relative", rf_rel_pred)]:
    baseline_results.loc[name] = [
        mean_absolute_error(y_val, pred),
        root_mean_squared_error(y_val, pred),
    ]
print("scale:", round(scale, 3))
baseline_results.round(5)

scale: 0.855


,MAE,RMSE
baseline,,
constant,0.01673,0.01925
vol_20,0.01314,0.01723
base_rv_5,0.01429,0.01922
random_forest,0.01325,0.01697
vol_20_scaled,0.01185,0.01646
rf_relative,0.01228,0.01679


In [10]:
from sklearn.ensemble import HistGradientBoostingRegressor

gb_rel = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    min_samples_leaf=20,
    random_state=0,
)
gb_rel.fit(train[FEATURES], y_train_rel)
gb_rel_pred = val["vol_20"] * np.exp(gb_rel.predict(val[FEATURES]))

baseline_results.loc["gb_relative"] = [
    mean_absolute_error(y_val, gb_rel_pred),
    root_mean_squared_error(y_val, gb_rel_pred),
]
baseline_results.round(5)

,MAE,RMSE
baseline,,
constant,0.01673,0.01925
vol_20,0.01314,0.01723
base_rv_5,0.01429,0.01922
random_forest,0.01325,0.01697
vol_20_scaled,0.01185,0.01646
rf_relative,0.01228,0.01679
gb_relative,0.01470,0.01989


In [11]:
for name, model in [("rf_relative", rf_rel), ("gb_relative", gb_rel)]:
    train_pred = train["vol_20"] * np.exp(model.predict(train[FEATURES]))
    print(name,
          "train MAE:", round(mean_absolute_error(train["target_rv_5"], train_pred), 5),
          "| val MAE:", round(mean_absolute_error(y_val, gb_rel_pred if name == "gb_relative" else rf_rel_pred), 5))

rf_relative train MAE: 0.01071 | val MAE: 0.01228
gb_relative train MAE: 0.00348 | val MAE: 0.0147


In [12]:
from cryptosense.regimes import add_regimes

regimes = add_regimes(df).loc[val.index, ["trend", "vol_regime"]]

errors = pd.DataFrame({
    "vol_20": (val["vol_20"] - y_val).abs(),
    "vol_20_scaled": (vol_20_scaled - y_val).abs(),
    "rf_relative": (rf_rel_pred - y_val).abs(),
}, index=val.index).join(regimes)

print(errors.groupby("trend")[["vol_20", "vol_20_scaled", "rf_relative"]].mean().round(5))
print(errors.groupby("vol_regime")[["vol_20", "vol_20_scaled", "rf_relative"]].mean().round(5))
print(errors.groupby("trend").size())

        vol_20  vol_20_scaled  rf_relative
trend                                     
down   0.01599        0.01427      0.01412
range  0.01102        0.01002      0.00968
up     0.01195        0.01087      0.01288
             vol_20  vol_20_scaled  rf_relative
vol_regime                                     
high        0.01525        0.01211      0.01205
low         0.01200        0.01171      0.01240
trend
down     270
range    241
up       214
dtype: int64


In [13]:
trainval = data.loc[:"2023-12-31"].iloc[:-HORIZON]
y_test = test["target_rv_5"]

y_tv_rel = np.log(trainval["target_rv_5"] / trainval["vol_20"])
scale_tv = np.exp(y_tv_rel.mean())

rf_final = RandomForestRegressor(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)
rf_final.fit(trainval[FEATURES], y_tv_rel)

test_forecasts = {
    "constant": np.full(len(test), trainval["target_rv_5"].mean()),
    "vol_20": test["vol_20"],
    "base_rv_5": test["base_rv_5"],
    "vol_20_scaled": test["vol_20"] * scale_tv,
    "rf_relative": test["vol_20"] * np.exp(rf_final.predict(test[FEATURES])),
}

test_results = pd.DataFrame(
    {name: {"MAE": mean_absolute_error(y_test, f), "RMSE": root_mean_squared_error(y_test, f)}
     for name, f in test_forecasts.items()}
).T

print("trainval:", trainval.index[0].date(), "->", trainval.index[-1].date(), len(trainval), "| scale:", round(scale_tv, 3))
test_results.round(5)

trainval: 2017-11-24 -> 2023-12-26 2224 | scale: 0.84


,MAE,RMSE
constant,0.01356,0.01555
vol_20,0.00956,0.01268
base_rv_5,0.01025,0.01386
vol_20_scaled,0.00870,0.01224
rf_relative,0.00818,0.01145


In [14]:
test_errors = pd.DataFrame({
    name: (pd.Series(f, index=test.index) - y_test).abs()
    for name, f in test_forecasts.items() if name in ["vol_20", "vol_20_scaled", "rf_relative"]
})
test_errors.groupby(test_errors.index.year).mean().round(5)

,vol_20,vol_20_scaled,rf_relative
date,,,
2024,0.01026,0.00944,0.00899
2025,0.00854,0.00781,0.00780
2026,0.00995,0.00889,0.00763


## Phase 4 findings: 5-day volatility forecasting (BTC/USDT)

**Setup:** Target = realized volatility of the next 5 daily log returns (root mean square, no demeaning). Chronological split: train to 2021, validation 2022–2023, test 2024+, with a 5-day purge before each boundary. Settings were chosen on validation; the chosen forecasts were refitted on train + validation and evaluated once on test.

**Observation:** On validation, scaled `vol_20` (`vol_20` multiplied by a factor learned from training) had the lowest MAE (0.01185, 9.8% better than `vol_20`), ahead of a Random Forest predicting the log ratio of future to recent volatility (0.01228). A forest predicting absolute volatility (0.01325) was anchored to the training era: its average forecast was 21% above the actual validation level. Gradient boosting overfit, with a training MAE of 0.0035 against a validation MAE of 0.0147. On the test set, the relative forest had the lowest MAE (0.00818, 14.4% better than `vol_20` and 6% better than scaled `vol_20`); by year, it beat scaled `vol_20` in 2024 and 2026 and tied in 2025. Scaled `vol_20` beat raw `vol_20` in every validation and test year, by 8–11%.

**Interpretation:** Recent volatility carries most of the predictable information about next week's volatility. Predicting volatility relative to its recent level removes the era drift that affects absolute targets. The relative forest and scaled `vol_20` are comparable: their ranking flipped between validation and test, so neither is reliably better. Errors were largest in downtrends, and the forest did worst in uptrends on validation, possibly because training-era uptrends were far more volatile than recent ones.

**Limitations:** Single asset (BTC); a single train/validation/test split rather than walk-forward evaluation; overlapping 5-day targets make each period contain fewer independent observations than its day count suggests; the test set covers under three years, including a partial 2026.